# Lab 07 solution, notebook 1: give the adventure planner a budget tool

The worked answer to the challenge of `07a_meet_the_agents_sdk.ipynb`. The setup repeats the notebook's tools, so
this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from agents import set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

In [3]:
import random
import requests
from agents import Agent, Runner, function_tool


@function_tool
def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo."""
    print(f"🌤️ TOOL CALLED: get_weather({city})")
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=10)
    locations = geo.json().get("results", [])
    if not locations:
        return f"City not found: {city}"
    location = locations[0]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": location["latitude"], "longitude": location["longitude"],
                "current": "temperature_2m,apparent_temperature,precipitation,wind_speed_10m"},
        timeout=10,
    )
    return str({"city": location["name"], "country": location["country"], **weather.json()["current"]})


@function_tool
def suggest_activity(weather: str) -> str:
    """Suggest an activity based on weather conditions."""
    print(f"🎯 TOOL CALLED: suggest_activity({weather})")
    return ("Sunny: hiking, cycling, picnic. Rainy: museum, board games, cooking. "
            "Cold: cafe, indoor climbing, cinema. Hot: swimming, indoor activities.")


@function_tool
def get_random_challenge() -> str:
    """Generate a random fun travel challenge."""
    print("🎲 TOOL CALLED: get_random_challenge()")
    return random.choice(["Take a photo of something shaped like a heart.",
                          "Try a local food you've never eaten.",
                          "Learn three words in a new language.",
                          "Find the most unusual building nearby."])

## The budget tool

`estimate_budget` looks an activity up in a small dictionary of made-up price ranges. `activity.lower()` makes the
lookup ignore capital letters, and `.get(..., default)` returns a general range for anything the dictionary does not
list. The prices are invented for the exercise.

In [4]:
PRICES = {
    "hiking": "free", "cycling": "10 to 20 USD for a rental", "picnic": "15 to 25 USD",
    "museum": "10 to 20 USD", "board games": "5 to 15 USD at a cafe", "cooking": "40 to 60 USD for a class",
    "cafe": "5 to 15 USD", "indoor climbing": "20 to 30 USD", "cinema": "10 to 15 USD",
    "swimming": "5 to 20 USD",
}


@function_tool
def estimate_budget(activity: str) -> str:
    """Estimate what an activity costs per person."""
    print(f"💰 TOOL CALLED: estimate_budget({activity})")
    return PRICES.get(activity.lower(), "20 to 50 USD")

The planner gets the fourth tool and one more line of instructions, which says when to use it.

In [5]:
budget_planner = Agent(
    name="Adventure Planner",
    instructions=(
        "You are a creative adventure planner. "
        "Always check current weather before recommending activities. "
        "Use suggest_activity for activity ideas. "
        "Use estimate_budget for the cost of the one activity you recommend. "
        "Use get_random_challenge to add a fun challenge. "
        "Give a short, practical plan."
    ),
    model=MODEL,
    tools=[get_weather, suggest_activity, estimate_budget, get_random_challenge],
)

result = await Runner.run(budget_planner, "I'm in Lisbon. Plan a fun afternoon, tell me what it costs, and give me a challenge!")

print(result.final_output)
print()
print("order of tool calls:", [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"])

🌤️ TOOL CALLED: get_weather(Lisbon)


🎯 TOOL CALLED: suggest_activity(Lisbon: 18.8°C, feels like 18.8°C, light precipitation (0.1 mm), breezy with 18.9 km/h wind.)


🎲 TOOL CALLED: get_random_challenge()


💰 TOOL CALLED: estimate_budget(Visit a museum in Lisbon)


**Lisbon afternoon plan:** It’s about **19°C**, breezy, with a little drizzle—good weather for an indoor outing. Visit a museum, then take a relaxed café stop nearby. Check the museum’s closing time before heading out, as it’s already late afternoon.

**Cost:** Estimate **$20–$50 per person** for the museum visit.

**Challenge:** Learn three Portuguese words today—try *obrigado/obrigada* (thank you), *por favor* (please), and *saudade* (a deep feeling of longing).

order of tool calls: ['get_weather', 'suggest_activity', 'get_random_challenge', 'estimate_budget']


Four `TOOL CALLED` lines, with `get_weather` first. `estimate_budget` came last, after the model had chosen an
activity from what `suggest_activity` returned.

**Notice** the argument: `Visit a museum in Lisbon`. Our dictionary has the key `museum`, so the exact lookup found
nothing and the tool returned the default range, 20 to 50 USD, which the plan then quoted. The tool did what we
wrote. The model wrote the argument in its own words, as it always does. A tool that expects one of a fixed list of
values should say so in its description, or accept free text and match it more loosely.